# GPRA Year 2 Scoring Pipeline

This notebook automates the scoring and reporting of **Government Performance and Results Act (GPRA)** measures for two federal school mental health grant programs:

- **MHSP** — Mental Health Service Professionals grant program (higher education grantees)
- **SBMH** — School-Based Mental Health grant program (K–12 education agencies)

### What this notebook does
1. Loads and standardizes raw APR (Annual Performance Report) data from each program
2. Cleans GPRA measure columns (handling ratio strings, missing value codes, and type coercion)
3. Scores each grantee by comparing actual vs. target values across all GPRA measures
4. Identifies grantees meeting at least 2 of their performance targets
5. Computes program-level and overall summary statistics
6. Exports intermediate outputs (CSV, JSON) for reproducibility and downstream reporting

### Data
Input data are synthetic and anonymized for portfolio purposes. The schema mirrors the structure of real federal APR Smartsheet exports.

---

## 1. Setup

Load libraries, configure logging, and define file paths.

- **Logging** is configured to write to both a `.log` file and the console for full traceability.
- **File paths** use `os.path.join` with a configurable `DATA_DIR` — update this to point to your local data directory.

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
import logging
import os
import json

# ── Logging setup ─────────────────────────────────────────────────────────────
# Clear any pre-existing handlers to avoid duplicate log entries on re-run
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("gpra2_scoring.log"),
        logging.StreamHandler()
    ]
)

# ── File paths ────────────────────────────────────────────────────────────────
# Update DATA_DIR to point to your local data folder
DATA_DIR = os.path.join(os.getcwd(), "data")

sbmh_file_path = os.path.join(DATA_DIR, "FINAL_Year2_SBMH_APR.xlsx")
mhsp_file_path = os.path.join(DATA_DIR, "FINAL_Year2_MHSP_APR.xlsx")

logging.info("Setup complete. File paths and logging configured.")

## 2. Load Data

Read the first sheet from each APR Excel file into a DataFrame.
Each file represents one grant program's Year 2 reporting data.

In [ ]:
sbmh_2025 = pd.read_excel(sbmh_file_path, sheet_name=0)
mhsp_2025 = pd.read_excel(mhsp_file_path, sheet_name=0)

print(f"SBMH: {sbmh_2025.shape[0]} grantees, {sbmh_2025.shape[1]} columns")
print(f"MHSP: {mhsp_2025.shape[0]} grantees, {mhsp_2025.shape[1]} columns")

logging.info(f"Loaded SBMH data from {sbmh_file_path}")
logging.info(f"Loaded MHSP data from {mhsp_file_path}")

## 3. Standardize Column Names

Rename raw Smartsheet column headers to consistent `snake_case` names.

- SBMH measures: GPRA 1–5 (Hired, Retention, Ratio, Attrition, Students Served)
- MHSP measures: GPRA 1A/1B–3A/3B (paired target/actual per measure)

The `_y2` suffix marks these as Year 2 values to support multi-year longitudinal work downstream.

In [ ]:
sbmh_2025 = sbmh_2025.rename(columns={
    'Grant ID': 'grant_id',
    'Grant Name': 'grantee_name',
    'GPRA 1 (Hired) Target (Raw)': 'gpra_1_target_y2',
    'GPRA 1 (Hired) Actual (Raw)': 'gpra_1_actual_y2',
    'GPRA 2 (Retention) Target (Raw)': 'gpra_2_target_y2',
    'GPRA 2 (Retention) Actual (Raw)': 'gpra_2_actual_y2',
    'GPRA 3 (Ratio) Target': 'gpra_3_target_y2_ratio',
    'GPRA 3 (Ratio) Actual': 'gpra_3_actual_y2_ratio',
    'GPRA 4 (Attrition) Target (Raw)': 'gpra_4_target_y2',
    'GPRA 4 (Attrition) Actual (Raw)': 'gpra_4_actual_y2',
    'GPRA 5 (Students Served) Target (Raw)': 'gpra_5_target_y2',
    'GPRA 5 (Students Served) Actual (Raw)': 'gpra_5_actual_y2'
})

mhsp_2025 = mhsp_2025.rename(columns={
    'Grant Number': 'grant_id',
    'Grant Name': 'grantee_name',
    'GPRA 1A Target (Raw)': 'gpra_1a_target_y2',
    'GPRA 1A Actual (Raw)': 'gpra_1a_actual_y2',
    'GPRA 1B Target (Raw)': 'gpra_1b_target_y2',
    'GPRA 1B Actual (Raw)': 'gpra_1b_actual_y2',
    'GPRA 2A Target (Raw)': 'gpra_2a_target_y2',
    'GPRA 2A Actual (Raw)': 'gpra_2a_actual_y2',
    'GPRA 2B Target (Raw)': 'gpra_2b_target_y2',
    'GPRA 2B Actual (Raw)': 'gpra_2b_actual_y2',
    'GPRA 3A Target (Raw)': 'gpra_3a_target_y2',
    'GPRA 3A Actual (Raw)': 'gpra_3a_actual_y2',
    'GPRA 3B Target (Raw)': 'gpra_3b_target_y2',
    'GPRA 3B Actual (Raw)': 'gpra_3b_actual_y2'
})

print("Column renaming complete.")

## 4. Data Cleaning

Two custom cleaning functions handle the messy data formats common in Smartsheet exports:

- **`convert_ratio_to_decimal`**: GPRA 3 (student-to-provider ratio) is reported as a string like `"49,786/256"`. This function parses the numerator and denominator and returns `larger / smaller` as a decimal.
- **`clean_and_convert`**: Replaces non-numeric sentinel values (`999`, `n/a`, `NR`, etc.) with `np.nan` and coerces everything else to float. These codes appear when grantees did not report or the measure was not applicable.

In [ ]:
def convert_ratio_to_decimal(ratio_str):
    """
    Convert a ratio string (e.g. '49,786/256' or '200/1') to a decimal.
    Returns larger / smaller to normalize direction.
    Returns np.nan for missing or unparseable input.
    """
    if pd.isna(ratio_str):
        return np.nan
    ratio_str = str(ratio_str).replace(',', '')
    if '/' in ratio_str:
        parts = ratio_str.split('/')
        if len(parts) == 2:
            try:
                num, den = float(parts[0]), float(parts[1])
                if num == 0 or den == 0:
                    return 0.0
                return max(num, den) / min(num, den)
            except ValueError:
                return np.nan
    try:
        return float(ratio_str)
    except ValueError:
        return np.nan


def clean_and_convert(x):
    """
    Coerce a value to float, replacing known missing-data codes with np.nan.
    Codes treated as missing: 999, n/a, na, nr (case-insensitive).
    """
    if pd.isna(x):
        return np.nan
    x_str = str(x).strip().lower()
    if x_str in {'999', 'n/a', 'na', 'nr'}:
        return np.nan
    try:
        return float(x)
    except ValueError:
        return np.nan


# Apply ratio conversion for SBMH GPRA 3
sbmh_2025['gpra_3_target_y2'] = sbmh_2025['gpra_3_target_y2_ratio'].apply(convert_ratio_to_decimal)
sbmh_2025['gpra_3_actual_y2'] = sbmh_2025['gpra_3_actual_y2_ratio'].apply(convert_ratio_to_decimal)

# Clean all numeric GPRA columns for SBMH
sbmh_gpra_cols = [
    'gpra_1_target_y2', 'gpra_1_actual_y2',
    'gpra_2_target_y2', 'gpra_2_actual_y2',
    'gpra_3_target_y2', 'gpra_3_actual_y2',
    'gpra_4_target_y2', 'gpra_4_actual_y2',
    'gpra_5_target_y2', 'gpra_5_actual_y2'
]
for col in sbmh_gpra_cols:
    sbmh_2025[col] = sbmh_2025[col].apply(clean_and_convert)

# Clean all numeric GPRA columns for MHSP
mhsp_gpra_cols = [
    'gpra_1a_target_y2', 'gpra_1a_actual_y2',
    'gpra_1b_target_y2', 'gpra_1b_actual_y2',
    'gpra_2a_target_y2', 'gpra_2a_actual_y2',
    'gpra_2b_target_y2', 'gpra_2b_actual_y2',
    'gpra_3a_target_y2', 'gpra_3a_actual_y2',
    'gpra_3b_target_y2', 'gpra_3b_actual_y2'
]
for col in mhsp_gpra_cols:
    mhsp_2025[col] = mhsp_2025[col].apply(clean_and_convert)

print("Cleaning complete.")
print("\nSBMH GPRA 3 ratio → decimal preview:")
sbmh_2025[['gpra_3_target_y2_ratio', 'gpra_3_target_y2',
           'gpra_3_actual_y2_ratio', 'gpra_3_actual_y2']].head()

## 5. Save Cleaned Data

Write cleaned DataFrames to Excel for downstream analysis and audit trail.

In [ ]:
output_dir = os.path.join(os.getcwd(), "output")
os.makedirs(output_dir, exist_ok=True)

sbmh_cleaned_path = os.path.join(output_dir, "SBMH_2025_Cleaned.xlsx")
mhsp_cleaned_path = os.path.join(output_dir, "MHSP_2025_Cleaned.xlsx")

sbmh_2025.to_excel(sbmh_cleaned_path, index=False)
mhsp_2025.to_excel(mhsp_cleaned_path, index=False)

logging.info(f"Cleaned SBMH data saved to {sbmh_cleaned_path}")
logging.info(f"Cleaned MHSP data saved to {mhsp_cleaned_path}")
print("Cleaned files saved.")

## 6. Score Grantees: Met vs. Not Met

For each grantee, count how many GPRA targets were met (actual ≥ target).

**SBMH exception:** GPRA 4 measures attrition (lower is better), so it is met when `actual ≤ target`.

A grantee is considered to be **meeting goals** if they met at least 2 of their GPRA targets.

In [ ]:
# ── MHSP scoring ─────────────────────────────────────────────────────────────
mhsp_actual_cols = ['gpra_1a_actual_y2', 'gpra_1b_actual_y2',
                    'gpra_2a_actual_y2', 'gpra_2b_actual_y2',
                    'gpra_3a_actual_y2', 'gpra_3b_actual_y2']
mhsp_target_cols = ['gpra_1a_target_y2', 'gpra_1b_target_y2',
                    'gpra_2a_target_y2', 'gpra_2b_target_y2',
                    'gpra_3a_target_y2', 'gpra_3b_target_y2']

mhsp_met_mask = (
    (mhsp_2025[mhsp_actual_cols].values >= mhsp_2025[mhsp_target_cols].values)
    & ~mhsp_2025[mhsp_actual_cols].isna().values
    & ~mhsp_2025[mhsp_target_cols].isna().values
)
mhsp_2025['meeting_targets'] = mhsp_met_mask.sum(axis=1)
rows_meeting_targets_mhsp = mhsp_2025[mhsp_2025['meeting_targets'] >= 2]

# ── SBMH scoring ─────────────────────────────────────────────────────────────
sbmh_actual_cols = ['gpra_1_actual_y2', 'gpra_2_actual_y2',
                    'gpra_3_actual_y2', 'gpra_4_actual_y2', 'gpra_5_actual_y2']
sbmh_target_cols = ['gpra_1_target_y2', 'gpra_2_target_y2',
                    'gpra_3_target_y2', 'gpra_4_target_y2', 'gpra_5_target_y2']

# Default: actual >= target
sbmh_met_mask = (
    (sbmh_2025[sbmh_actual_cols].values >= sbmh_2025[sbmh_target_cols].values)
    & ~sbmh_2025[sbmh_actual_cols].isna().values
    & ~sbmh_2025[sbmh_target_cols].isna().values
)
# GPRA 4 (index 3) is attrition: met when actual <= target
sbmh_met_mask[:, 3] = (
    (sbmh_2025['gpra_4_actual_y2'].values <= sbmh_2025['gpra_4_target_y2'].values)
    & ~sbmh_2025['gpra_4_actual_y2'].isna().values
    & ~sbmh_2025['gpra_4_target_y2'].isna().values
)
sbmh_2025['meeting_targets'] = sbmh_met_mask.sum(axis=1)
rows_meeting_targets_sbmh = sbmh_2025[sbmh_2025['meeting_targets'] >= 2]

print(f"MHSP grantees meeting ≥2 targets: {len(rows_meeting_targets_mhsp)} / {len(mhsp_2025)}")
print(f"SBMH grantees meeting ≥2 targets: {len(rows_meeting_targets_sbmh)} / {len(sbmh_2025)}")

## 7. Validation Preview

Inspect per-grantee target-met flags alongside raw actual and target values for a sanity check.

In [ ]:
from IPython.display import display

# MHSP validation table
mhsp_met_df = pd.DataFrame(
    mhsp_met_mask,
    columns=[col.replace('_actual_y2', '_met') for col in mhsp_actual_cols]
)
mhsp_validation = pd.concat([
    mhsp_2025[['grant_id', 'grantee_name'] + mhsp_actual_cols + mhsp_target_cols],
    mhsp_met_df,
    mhsp_2025[['meeting_targets']]
], axis=1)
print("MHSP Validation Preview (first 5 grantees):")
display(mhsp_validation.head())

# SBMH validation table
sbmh_met_df = pd.DataFrame(
    sbmh_met_mask,
    columns=[col.replace('_actual_y2', '_met') for col in sbmh_actual_cols]
)
sbmh_validation = pd.concat([
    sbmh_2025[['grant_id', 'grantee_name'] + sbmh_actual_cols + sbmh_target_cols],
    sbmh_met_df,
    sbmh_2025[['meeting_targets']]
], axis=1)
print("\nSBMH Validation Preview (first 5 grantees):")
display(sbmh_validation.head())

## 8. Summary Statistics

Calculate program-level and overall percentages of grantees meeting goals.

In [ ]:
n_mhsp_met = len(rows_meeting_targets_mhsp)
n_sbmh_met = len(rows_meeting_targets_sbmh)
total_mhsp = len(mhsp_2025)
total_sbmh = len(sbmh_2025)

percentage_mhsp = (n_mhsp_met / total_mhsp) * 100
percentage_sbmh = (n_sbmh_met / total_sbmh) * 100
percentage_grantees = ((n_mhsp_met + n_sbmh_met) / (total_mhsp + total_sbmh)) * 100

print(f"MHSP: {n_mhsp_met}/{total_mhsp} grantees meeting ≥2 targets ({percentage_mhsp:.2f}%)")
print(f"SBMH: {n_sbmh_met}/{total_sbmh} grantees meeting ≥2 targets ({percentage_sbmh:.2f}%)")
print(f"Overall: {n_mhsp_met + n_sbmh_met}/{total_mhsp + total_sbmh} ({percentage_grantees:.2f}%)")

logging.info(f"Percentage of grantees meeting at least 2 targets: {percentage_grantees:.2f}%")

## 9. Export Outputs

Save grantee-level results and summary statistics for audit and downstream reporting.

In [ ]:
# Save grantee-level outputs
rows_meeting_targets_mhsp.to_csv(os.path.join(output_dir, "MHSP_Grantees_Meeting_Targets.csv"), index=False)
rows_meeting_targets_sbmh.to_csv(os.path.join(output_dir, "SBMH_Grantees_Meeting_Targets.csv"), index=False)

# Save summary statistics to JSON
summary_stats = {
    "percentage_grantees_meeting_goals": percentage_grantees,
    "percentage_mhsp_meeting_targets": percentage_mhsp,
    "percentage_sbmh_meeting_targets": percentage_sbmh,
    "mhsp_grantees_meeting_targets": n_mhsp_met,
    "sbmh_grantees_meeting_targets": n_sbmh_met,
    "total_mhsp": total_mhsp,
    "total_sbmh": total_sbmh
}
with open(os.path.join(output_dir, "gpra_summary_stats.json"), "w") as f:
    json.dump(summary_stats, f, indent=2)

logging.info("Intermediate outputs saved for reproducibility.")
logging.info("GPRA 2 scoring script completed successfully.")
print("All outputs saved.")